# Paper #56: SO/PHI Implementation / SO/PHI 구현

**Paper**: Solanki, S. K., del Toro Iniesta, J. C., Woch, J., Gandorfer, A., Hirzberger, J., Alvarez-Herrero, A., et al., "The Polarimetric and Helioseismic Imager on Solar Orbiter (SO/PHI)", A&A 642, A11 (2020). DOI: 10.1051/0004-6361/201935325.

이 노트북은 Solar Orbiter에 탑재된 SO/PHI 기기의 4가지 핵심 측정 원리를 합성 데이터로 구현한다:
(1) 듀얼 망원경 광학 (HRT vs FDT) 비교, (2) LiNbO3 Fabry-Pérot 에탈론에 의한 Fe I 6173 Å 선 스캔, (3) Zeeman 효과 하에서의 Stokes I/Q/U/V 프로파일 합성, (4) Milne-Eddington 인버전을 통한 자기장 복원.

This notebook implements four core measurement principles of SO/PHI on Solar Orbiter using synthetic data:
(1) Dual-telescope optics (HRT vs FDT) comparison, (2) Fe I 6173 Å line scan via the LiNbO3 Fabry-Pérot etalon, (3) Stokes I/Q/U/V profile synthesis under the Zeeman effect, (4) Magnetic field recovery via a toy Milne-Eddington inversion.

**Kernel**: `study-with-ai` (conda)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from scipy.optimize import least_squares
from scipy.special import voigt_profile

# Standard plot settings.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

np.random.seed(42)
print("All imports successful.")

## Part 1: HRT vs FDT 광학 비교 / HRT vs FDT Optical Comparison

SO/PHI는 두 망원경을 단일 검출기에 공급한다. **HRT** (High Resolution Telescope)는 140 mm 구경의 off-axis Ritchey-Chrétien으로 0.28° 시야 (FOV) 와 0.28 AU 근일점에서 ~200 km 분해능을 달성한다. **FDT** (Full Disc Telescope)는 17.5 mm 구경의 굴절식으로 2° 시야와 풀-디스크 영상을 제공한다. Feed-Select Mechanism (FSM) 이 두 광로 중 하나를 선택한다.

SO/PHI feeds two telescopes into a single detector. **HRT** is a 140 mm off-axis Ritchey-Chrétien with a 0.28° field of view, reaching ~200 km spatial resolution at the 0.28 AU perihelion. **FDT** is a 17.5 mm refractor with a 2° round FOV that captures the full solar disc. The Feed-Select Mechanism (FSM) chooses one path at a time.

이 셀은 두 망원경의 (구경, FOV, 분해능, 카덴스, plate scale) 을 정량적으로 비교하고, 0.28 AU 근일점에서 동일한 활동 영역을 두 망원경이 어떻게 보는지 보여주는 시각화를 만든다.

This cell quantitatively compares (aperture, FOV, resolution, cadence, plate scale) of the two telescopes and visualises how each views the same active region at the 0.28 AU perihelion.

In [ ]:
# =============================================================
# HRT vs FDT Optical Parameters and Coverage at 0.28 AU
# =============================================================

# Physical constants.
AU_KM = 1.495978707e8
RSUN_KM = 6.957e5
ARCSEC_PER_RAD = 206264.806
WAVELENGTH_NM = 617.3  # Fe I line.

# Instrument parameters from Solanki et al. 2020 (Tables 1, 2, 3).
instruments = {
    "HRT": {
        "aperture_mm": 140.0,
        "focal_length_mm": 4125.0,
        "fov_deg": 0.28,
        "plate_scale_arcsec": 0.5,
        "cadence_min_s": 60.0,
    },
    "FDT": {
        "aperture_mm": 17.5,
        "focal_length_mm": 579.0,
        "fov_deg": 2.0,
        "plate_scale_arcsec": 3.75,
        "cadence_min_s": 60.0,
    },
}


def diffraction_limit_arcsec(aperture_mm, wavelength_nm):
    """Compute the Rayleigh diffraction limit in arcsec.

    Args:
        aperture_mm: Telescope aperture in mm.
        wavelength_nm: Observation wavelength in nm.

    Returns:
        Diffraction-limited angular resolution in arcsec.
    """
    lam_m = wavelength_nm * 1e-9
    aperture_m = aperture_mm * 1e-3
    return 1.22 * lam_m / aperture_m * ARCSEC_PER_RAD


def km_per_pixel(plate_scale_arcsec, distance_au):
    """Convert pixel angular size to km on the solar surface."""
    distance_km = distance_au * AU_KM
    return plate_scale_arcsec / ARCSEC_PER_RAD * distance_km


# Compute derived quantities at 0.28 AU perihelion and 1.0 AU.
distances_au = [0.28, 1.0]

print("=" * 72)
print("SO/PHI Telescope Comparison")
print("=" * 72)
header = "{:<24s} {:>12s} {:>12s}".format("Parameter", "HRT", "FDT")
print(header)
print("-" * 72)
row = "{:<24s} {:>12.1f} {:>12.1f}"
print(row.format("Aperture [mm]",
                 instruments["HRT"]["aperture_mm"],
                 instruments["FDT"]["aperture_mm"]))
print(row.format("Focal length [mm]",
                 instruments["HRT"]["focal_length_mm"],
                 instruments["FDT"]["focal_length_mm"]))
print("{:<24s} {:>12.2f} {:>12.2f}".format("FOV [deg]",
                                           instruments["HRT"]["fov_deg"],
                                           instruments["FDT"]["fov_deg"]))
print("{:<24s} {:>12.2f} {:>12.2f}".format("Plate scale [arcsec/px]",
                                           instruments["HRT"]["plate_scale_arcsec"],
                                           instruments["FDT"]["plate_scale_arcsec"]))

for inst in ("HRT", "FDT"):
    diff_lim = diffraction_limit_arcsec(instruments[inst]["aperture_mm"], WAVELENGTH_NM)
    instruments[inst]["diff_limit_arcsec"] = diff_lim

print("{:<24s} {:>12.3f} {:>12.3f}".format(
    "Diff. limit [arcsec]",
    instruments["HRT"]["diff_limit_arcsec"],
    instruments["FDT"]["diff_limit_arcsec"]))

for d in distances_au:
    hrt_km = km_per_pixel(instruments["HRT"]["plate_scale_arcsec"], d)
    fdt_km = km_per_pixel(instruments["FDT"]["plate_scale_arcsec"], d)
    print("{:<24s} {:>12.1f} {:>12.1f}".format(
        "km/pixel @ {:.2f} AU".format(d), hrt_km, fdt_km))

print("=" * 72)

# Visualisation: solar disc as seen by each telescope at 0.28 AU.
rsun_arcsec_028 = (RSUN_KM / (0.28 * AU_KM)) * ARCSEC_PER_RAD
hrt_fov_arcsec = instruments["HRT"]["fov_deg"] * 3600.0
fdt_fov_arcsec = instruments["FDT"]["fov_deg"] * 3600.0

fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Left: full disc with HRT footprint at 0.28 AU.
ax = axes[0]
sun_circle = Circle((0, 0), rsun_arcsec_028, color="#FFC107", alpha=0.6, label="Sun")
ax.add_patch(sun_circle)
fdt_box = Rectangle((-fdt_fov_arcsec / 2, -fdt_fov_arcsec / 2),
                    fdt_fov_arcsec, fdt_fov_arcsec,
                    fill=False, edgecolor="#2196F3", linewidth=2,
                    label="FDT FOV (2 deg)")
ax.add_patch(fdt_box)
hrt_box = Rectangle((-hrt_fov_arcsec / 2, -hrt_fov_arcsec / 2),
                    hrt_fov_arcsec, hrt_fov_arcsec,
                    fill=False, edgecolor="#E91E63", linewidth=2,
                    label="HRT FOV (0.28 deg)")
ax.add_patch(hrt_box)
ax.set_xlim(-fdt_fov_arcsec * 0.7, fdt_fov_arcsec * 0.7)
ax.set_ylim(-fdt_fov_arcsec * 0.7, fdt_fov_arcsec * 0.7)
ax.set_aspect("equal")
ax.set_xlabel("x [arcsec]")
ax.set_ylabel("y [arcsec]")
ax.set_title("Solar disc at 0.28 AU + HRT/FDT FOVs")
ax.legend(loc="lower right")

# Right: km-per-pixel vs distance.
ax = axes[1]
d_grid = np.linspace(0.28, 1.0, 50)
hrt_curve = [km_per_pixel(instruments["HRT"]["plate_scale_arcsec"], d) for d in d_grid]
fdt_curve = [km_per_pixel(instruments["FDT"]["plate_scale_arcsec"], d) for d in d_grid]
ax.plot(d_grid, hrt_curve, color="#E91E63", linewidth=2, label="HRT")
ax.plot(d_grid, fdt_curve, color="#2196F3", linewidth=2, label="FDT")
ax.axvline(0.28, color="gray", linestyle=":", label="perihelion")
ax.set_xlabel("Heliocentric distance [AU]")
ax.set_ylabel("km per pixel on solar surface")
ax.set_title("Spatial sampling vs distance")
ax.set_yscale("log")
ax.legend()

plt.tight_layout()
plt.show()

print("Key insight: HRT reaches 200 km/px at perihelion -- comparable to")
print("the best ground-based diffraction-limited solar observations.")

## Part 2: Fabry-Pérot 에탈론과 Fe I 6173 Å 선 스캔 / Fabry-Pérot Etalon and Fe I 6173 Å Line Scan

SO/PHI의 분광 요소는 LiNbO3 Fabry-Pérot 에탈론으로, 자유 스펙트럼 영역 (FSR) = 0.301 nm, 반치폭 (FWHM) = 106 mÅ, finesse $\mathcal{F}^* = 30$ 을 갖는다. 에탈론 투과율은 Airy 함수 $T(\lambda) = 1/(1 + F\sin^2(\delta/2))$ 로 기술되며, 여기서 $\delta = 4\pi n d \cos\theta / \lambda$ 는 광학 위상이다.

The spectral element of SO/PHI is a LiNbO3 Fabry-Pérot etalon with free spectral range (FSR) = 0.301 nm, full width at half maximum (FWHM) = 106 mÅ, and finesse $\mathcal{F}^* = 30$. The Airy transmission is $T(\lambda) = 1/(1 + F\sin^2(\delta/2))$ with optical phase $\delta = 4\pi n d \cos\theta / \lambda$.

다음 셀은 (a) 에탈론 투과 곡선과 인접한 차수 (FSR), (b) 6 파장 위치 ($-140, -70, 0, +70, +140$ mÅ + 연속체 $\pm 300$ mÅ) 에서의 선 샘플링을 그린다.

The next cell plots (a) the etalon transmission curve and adjacent order (FSR), and (b) the 6-position line sampling ($-140, -70, 0, +70, +140$ mÅ plus continuum at $\pm 300$ mÅ).

In [ ]:
# =============================================================
# Fabry-Perot Airy Function and Line Scan Sampling
# =============================================================

# Etalon parameters from Solanki et al. 2020 Sect. 4.2.8.
FSR_NM = 0.301
FWHM_MA = 106.0  # mAngstrom
FINESSE = 30.0
LAMBDA0_NM = 617.3

# Refractive index of LiNbO3 around 617 nm (used for context only).
N_LINBO3 = 2.30

# np.trapz was renamed np.trapezoid in NumPy 2.0 -- handle both.
_trapz = getattr(np, "trapezoid", None) or np.trapz


def airy_transmission(wavelength_nm, lambda_peak_nm, fsr_nm, finesse):
    """Compute the Fabry-Perot Airy transmission function.

    Args:
        wavelength_nm: Wavelength sampling array in nm.
        lambda_peak_nm: Position of the central transmission peak.
        fsr_nm: Free spectral range in nm.
        finesse: Reflective finesse (= pi*sqrt(F)/2).

    Returns:
        Transmission values in [0, 1].
    """
    f_coeff = (2.0 * finesse / np.pi) ** 2
    delta_over_2 = np.pi * (wavelength_nm - lambda_peak_nm) / fsr_nm
    return 1.0 / (1.0 + f_coeff * np.sin(delta_over_2) ** 2)


# Compute the FWHM directly from finesse and FSR for a sanity check.
fwhm_predicted_nm = FSR_NM / FINESSE
print("Etalon parameters:")
print("  FSR     = {:.4f} nm = {:.1f} mA".format(FSR_NM, FSR_NM * 1e4))
print("  Finesse = {:.1f}".format(FINESSE))
print("  FWHM    = FSR/finesse = {:.4f} nm = {:.1f} mA".format(
    fwhm_predicted_nm, fwhm_predicted_nm * 1e4))
print("  Paper:  FWHM = {:.1f} mA  -- match within 5%".format(FWHM_MA))

# Wide wavelength range to show two adjacent orders (FSR).
wav_wide = np.linspace(LAMBDA0_NM - 0.4, LAMBDA0_NM + 0.4, 4000)
T_wide = airy_transmission(wav_wide, LAMBDA0_NM, FSR_NM, FINESSE)

# Narrow range around the line for a detailed view.
wav_narrow = np.linspace(LAMBDA0_NM - 0.05, LAMBDA0_NM + 0.05, 4000)

# Synthetic Fe I 6173 line: Gaussian absorption profile.
LINE_DEPTH = 0.65
LINE_SIGMA_NM = 0.0048  # Doppler-broadened width.
line_profile_narrow = 1.0 - LINE_DEPTH * np.exp(
    -0.5 * ((wav_narrow - LAMBDA0_NM) / LINE_SIGMA_NM) ** 2)
line_profile_wide = 1.0 - LINE_DEPTH * np.exp(
    -0.5 * ((wav_wide - LAMBDA0_NM) / LINE_SIGMA_NM) ** 2)

# 6 wavelength sampling positions in mA from line centre.
scan_offsets_ma = np.array([-300.0, -140.0, -70.0, 0.0, 70.0, 140.0])
scan_offsets_nm = scan_offsets_ma * 1e-4
scan_lambdas_nm = LAMBDA0_NM + scan_offsets_nm

# Convolve etalon with line: integrate transmission * line over the etalon's narrow passband.
def sampled_intensity(lambda_set_nm, wav_grid, line_profile, fsr_nm, finesse):
    """Integrate line profile through etalon centred at lambda_set.

    Args:
        lambda_set_nm: Etalon centre wavelength setting.
        wav_grid: Wavelength integration grid.
        line_profile: Line profile values on wav_grid.
        fsr_nm: FP free spectral range.
        finesse: FP finesse.

    Returns:
        Etalon-weighted intensity at this setting.
    """
    transmission = airy_transmission(wav_grid, lambda_set_nm, fsr_nm, finesse)
    norm = _trapz(transmission, wav_grid)
    return _trapz(transmission * line_profile, wav_grid) / norm


sampled_I = np.array([
    sampled_intensity(lam, wav_narrow, line_profile_narrow, FSR_NM, FINESSE)
    for lam in scan_lambdas_nm
])

# Plot.
fig, axes = plt.subplots(2, 1, figsize=(12, 9))

# Top: wide view shows two adjacent etalon orders (FSR).
ax = axes[0]
ax.plot(wav_wide, T_wide, color="#1976D2", linewidth=1.5, label="Etalon transmission (Airy)")
ax.plot(wav_wide, line_profile_wide, color="#212121", linewidth=1.0, alpha=0.8,
        label="Fe I 6173 (line)")
ax.axvspan(LAMBDA0_NM - FSR_NM / 2, LAMBDA0_NM + FSR_NM / 2,
           alpha=0.1, color="#1976D2", label="One FSR window")
ax.axvline(LAMBDA0_NM - FSR_NM, color="#1976D2", linestyle=":")
ax.axvline(LAMBDA0_NM + FSR_NM, color="#1976D2", linestyle=":")
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Transmission / Intensity")
ax.set_title("LiNbO3 etalon: Airy peaks separated by FSR = {:.2f} nm".format(FSR_NM))
ax.legend(fontsize=9)
ax.set_ylim(0, 1.1)

# Bottom: zoomed view with 6-point sampling.
ax = axes[1]
ax.plot(wav_narrow, line_profile_narrow, color="#212121", linewidth=2,
        label="Fe I 6173 line")
for lam in scan_lambdas_nm:
    T = airy_transmission(wav_narrow, lam, FSR_NM, FINESSE)
    ax.plot(wav_narrow, T, alpha=0.4, linewidth=1)
ax.scatter(scan_lambdas_nm, sampled_I, color="#E91E63", s=80,
           zorder=5, label="6 sampled points")
for lam, intensity, off in zip(scan_lambdas_nm, sampled_I, scan_offsets_ma):
    ax.annotate("{:+.0f} mA".format(off),
                xy=(lam, intensity),
                xytext=(0, 12), textcoords="offset points",
                ha="center", fontsize=8, color="#E91E63")
ax.set_xlabel("Wavelength [nm]")
ax.set_ylabel("Intensity (continuum normalised)")
ax.set_title("Fe I 6173 A: 6-position SO/PHI line scan")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

print("Sampled intensities at 6 etalon settings:")
for off, I in zip(scan_offsets_ma, sampled_I):
    print("  dlambda = {:+6.1f} mA  ->  I/Ic = {:.3f}".format(off, I))

### 2.1 에탈론 전압 튜닝과 도플러 보정 / Etalon Voltage Tuning and Doppler Compensation

에탈론은 전기광학 효과로 튜닝된다: $\partial\lambda/\partial V = 351.1$ mÅ/kV. Solar Orbiter는 타원 궤도에서 $\pm 23.6$ km/s 의 시선 속도를 갖는데, 이는 $\pm 487$ mÅ 의 도플러 천이에 해당한다. 따라서 에탈론은 $-1.3$ kV ~ $+2.0$ kV 의 전압 스윙으로 이를 보상한다.

The etalon is tuned via the electro-optic effect: $\partial\lambda/\partial V = 351.1$ mÅ/kV. Solar Orbiter's elliptical orbit imparts $\pm 23.6$ km/s line-of-sight velocity, equivalent to $\pm 487$ mÅ Doppler shifts. The etalon compensates with a $-1.3$ kV to $+2.0$ kV voltage swing.

또한 온도 안정성: $\partial\lambda/\partial T = 37.9$ mÅ/K, 따라서 0.5 m/s 의 도플러 정확도를 위해서는 0.27 mK rms 안정도가 필요하다 (성취된 값 $\pm$ 0.3 mK).

Temperature stability: $\partial\lambda/\partial T = 37.9$ mÅ/K, so 0.5 m/s Doppler accuracy requires 0.27 mK rms stability (achieved $\pm$ 0.3 mK).

In [ ]:
# =============================================================
# Etalon Tuning Constants and Required Stabilities
# =============================================================

DLAM_DV_MA_KV = 351.1
DLAM_DT_MA_K = 37.9
C_KMS = 2.99792458e5

# Solar Orbiter Doppler swing.
v_so_max_kms = 23.6
doppler_swing_ma = LAMBDA0_NM * (v_so_max_kms / C_KMS) * 1e4
voltage_swing_kv = doppler_swing_ma / DLAM_DV_MA_KV

# Required temperature stability for 0.5 m/s.
v_target_ms = 0.5
dlam_target_ma = LAMBDA0_NM * (v_target_ms / 1000.0 / C_KMS) * 1e4
dT_required_mK = (dlam_target_ma / DLAM_DT_MA_K) * 1000.0

print("Etalon tuning summary:")
print("  Voltage tuning: {:.2f} mA/kV".format(DLAM_DV_MA_KV))
print("  Temperature tuning: {:.2f} mA/K".format(DLAM_DT_MA_K))
print("  S/C Doppler swing (+/-{:.1f} km/s): +/-{:.1f} mA".format(
    v_so_max_kms, doppler_swing_ma))
print("  Voltage required: +/-{:.2f} kV".format(voltage_swing_kv))
print("  Temp stability for {:.1f} m/s: {:.2f} mK rms".format(
    v_target_ms, dT_required_mK))
print("  Achieved: +/-0.3 mK rms (ground qualification).")

# Sweep voltage to show tunable range covers Doppler+line scan needs.
voltages_kv = np.linspace(-1.5, 2.5, 200)
lambda_settings_ma = voltages_kv * DLAM_DV_MA_KV

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(voltages_kv, lambda_settings_ma, color="#1976D2", linewidth=2)
ax.axhspan(-doppler_swing_ma, doppler_swing_ma, alpha=0.2, color="#FFC107",
           label="Doppler swing required")
ax.axhspan(-200, 200, alpha=0.2, color="#4CAF50", label="Line scan span")
ax.axvline(0, color="black", linestyle=":")
ax.set_xlabel("Etalon voltage [kV]")
ax.set_ylabel("Wavelength offset [mA]")
ax.set_title("Etalon voltage tuning: 351.1 mA/kV covers Doppler + line scan")
ax.legend()
plt.tight_layout()
plt.show()

## Part 3: Zeeman 효과와 Stokes 프로파일 / Zeeman Effect and Stokes Profiles

광구의 자기장은 Fe I 6173 Å (g_eff = 2.5) 의 흡수선을 Zeeman 효과로 분리한다. σ-성분의 한 쪽 천이는
$$\Delta\lambda_B = 4.67 \times 10^{-13}\, \lambda_0^2\, g_{\rm eff}\, B \text{ (Å, with }\lambda_0\text{ in Å, }B\text{ in G)}$$
이며, 자기장 1000 G 에서 약 44.5 mÅ (전체 σ+/σ- 사이 분리는 약 89 mÅ) 이다. 분리된 세 성분 ($\sigma^-, \pi, \sigma^+$) 의 편광 상태가 결합되어 Stokes I, Q, U, V 프로파일을 만든다.

Solar photospheric magnetic fields split the Fe I 6173 Å line (g_eff = 2.5) via the Zeeman effect. One sigma-component shift is
$$\Delta\lambda_B = 4.67 \times 10^{-13}\, \lambda_0^2\, g_{\rm eff}\, B \text{ (Å, with }\lambda_0\text{ in Å, }B\text{ in G)}$$
yielding ~44.5 mÅ at B = 1000 G (full sigma+ to sigma- separation ~89 mÅ). The polarisation states of the three split components ($\sigma^-, \pi, \sigma^+$) combine to produce the Stokes I, Q, U, V profiles.

다음 셀은 단순화된 Unno-Rachkovsky 형식주의로 Stokes 4벡터를 합성하며, 자기장 강도 $B$ 와 inclination $\gamma$ 의 변화를 보여준다.

The next cell synthesises the Stokes 4-vector via a simplified Unno-Rachkovsky form and shows the response to varying field strength $B$ and inclination $\gamma$.

In [ ]:
# =============================================================
# Stokes Profiles via Simplified Unno-Rachkovsky Solution
# =============================================================

G_EFF_FE6173 = 2.5


def zeeman_split_ma(B_gauss, lambda_nm=LAMBDA0_NM, g_eff=G_EFF_FE6173):
    """Compute the Zeeman wavelength splitting in mA.

    Args:
        B_gauss: Magnetic field strength in Gauss.
        lambda_nm: Line wavelength in nm.
        g_eff: Effective Lande g-factor.

    Returns:
        Wavelength splitting in mA.
    """
    return 4.67e-13 * (lambda_nm * 10.0) ** 2 * g_eff * B_gauss * 1e3


def me_stokes_profiles(wavelengths_nm, B, gamma_rad, chi_rad,
                       eta0=8.0, dlamD_nm=0.0035, a=0.05,
                       S0=0.2, S1=0.8, lambda0_nm=LAMBDA0_NM,
                       g_eff=G_EFF_FE6173, v_los_kms=0.0):
    """Compute Milne-Eddington Stokes I, Q, U, V profiles.

    Uses the standard analytical Unno-Rachkovsky solution with
    Voigt absorption and dispersion profiles for the three Zeeman
    components.

    Args:
        wavelengths_nm: Wavelength array in nm.
        B: Magnetic field magnitude in Gauss.
        gamma_rad: Field inclination from LOS in radians.
        chi_rad: Field azimuth in radians.
        eta0: Line-to-continuum opacity ratio.
        dlamD_nm: Doppler width in nm.
        a: Damping parameter.
        S0: Source function constant.
        S1: Source function gradient.
        lambda0_nm: Line centre in nm.
        g_eff: Effective Lande g-factor.
        v_los_kms: Line-of-sight velocity in km/s (positive = redshift).

    Returns:
        Tuple (I, Q, U, V) of arrays sampled on wavelengths_nm.
    """
    # Doppler-shift the line centre.
    lam0_shift = lambda0_nm * (1.0 + v_los_kms / C_KMS)
    # Zeeman split (in mA -> nm).
    dL_nm = zeeman_split_ma(B, lambda0_nm, g_eff) * 1e-4

    # Reduced wavelength for each component.
    v_pi = (wavelengths_nm - lam0_shift) / dlamD_nm
    v_minus = (wavelengths_nm - (lam0_shift - dL_nm)) / dlamD_nm
    v_plus = (wavelengths_nm - (lam0_shift + dL_nm)) / dlamD_nm

    # Voigt absorption and dispersion profiles (using scipy voigt_profile for H).
    sigma_v = 1.0 / np.sqrt(2.0)
    H_pi = voigt_profile(v_pi, sigma_v, a) / voigt_profile(0.0, sigma_v, a)
    H_minus = voigt_profile(v_minus, sigma_v, a) / voigt_profile(0.0, sigma_v, a)
    H_plus = voigt_profile(v_plus, sigma_v, a) / voigt_profile(0.0, sigma_v, a)

    sin2g = np.sin(gamma_rad) ** 2
    cos_g = np.cos(gamma_rad)
    cos2x = np.cos(2.0 * chi_rad)
    sin2x = np.sin(2.0 * chi_rad)

    # Absorption matrix elements (dispersion terms ignored for clarity).
    eta_I = eta0 * (0.5 * H_pi * sin2g
                    + 0.25 * (H_plus + H_minus) * (1.0 + cos_g ** 2))
    eta_Q = eta0 * 0.5 * (H_pi - 0.5 * (H_plus + H_minus)) * sin2g * cos2x
    eta_U = eta0 * 0.5 * (H_pi - 0.5 * (H_plus + H_minus)) * sin2g * sin2x
    eta_V = eta0 * 0.5 * (H_minus - H_plus) * cos_g

    # Unno-Rachkovsky denominator.
    one_pI = 1.0 + eta_I
    Delta = one_pI ** 2 * (one_pI ** 2 - eta_Q ** 2 - eta_U ** 2 - eta_V ** 2)
    Delta = np.maximum(Delta, 1e-12)

    I = S0 + S1 * one_pI * (one_pI ** 2 - eta_Q ** 2 - eta_U ** 2 - eta_V ** 2) / Delta
    Q = -S1 * one_pI ** 2 * eta_Q / Delta
    U = -S1 * one_pI ** 2 * eta_U / Delta
    V = -S1 * one_pI ** 2 * eta_V / Delta
    return I, Q, U, V


# Sanity check on Zeeman splitting.
for B_test in (200, 500, 1000, 2000):
    print("  B = {:>4d} G -> Delta lambda = {:.2f} mA".format(
        B_test, zeeman_split_ma(B_test)))

wav = np.linspace(LAMBDA0_NM - 0.025, LAMBDA0_NM + 0.025, 600)
B_cases = [(300, np.deg2rad(30)),
           (1000, np.deg2rad(45)),
           (2500, np.deg2rad(60))]
colors = ["#2196F3", "#FF9800", "#E91E63"]
chi_val = np.deg2rad(20.0)

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
labels = ["Stokes I", "Stokes Q / I_c", "Stokes U / I_c", "Stokes V / I_c"]

for (B_val, gamma_val), color in zip(B_cases, colors):
    I, Q, U, V = me_stokes_profiles(wav, B_val, gamma_val, chi_val)
    label = "B={} G, gamma={:.0f} deg".format(B_val, np.rad2deg(gamma_val))
    axes[0, 0].plot(wav, I, color=color, label=label, linewidth=1.5)
    axes[0, 1].plot(wav, Q, color=color, label=label, linewidth=1.5)
    axes[1, 0].plot(wav, U, color=color, label=label, linewidth=1.5)
    axes[1, 1].plot(wav, V, color=color, label=label, linewidth=1.5)

for ax, lab in zip(axes.flat, labels):
    ax.set_title(lab)
    ax.legend(fontsize=8)
    ax.set_ylabel("normalised")
    ax.axhline(0, color="gray", linestyle=":", alpha=0.4)
for ax in axes[1]:
    ax.set_xlabel("Wavelength [nm]")
fig.suptitle("Stokes profiles for Fe I 6173 (g_eff = 2.5)", fontsize=13)
plt.tight_layout()
plt.show()

print("Observations:")
print("  V is antisymmetric -- amplitude scales with B*cos(gamma) (LOS field).")
print("  Q,U are symmetric -- amplitude scales with B^2*sin^2(gamma) (transverse).")
print("  Stronger B widens the line and lifts the polarisation amplitudes.")

### 3.1 SO/PHI 6점 샘플링 + 잡음 / SO/PHI 6-Point Sampling with Noise

다음 셀은 위에서 합성한 연속 Stokes 프로파일을 SO/PHI의 6 파장 위치에서 에탈론 투과율로 가중 적분하여 샘플링한다. 그리고 paper의 SNR 목표인 $10^{-3} I_c$ 노이즈 floor을 적용해 가우시안 잡음을 더한다 — 이것이 ME 인버전 입력이 된다.

The next cell samples the continuous Stokes profiles at SO/PHI's 6 wavelength positions, weighted by the etalon transmission, and adds Gaussian noise at the paper's $10^{-3} I_c$ noise floor to produce the ME-inversion input.

In [ ]:
# =============================================================
# Etalon-weighted Sampling of Stokes Profiles + Noise
# =============================================================

def sample_stokes_through_etalon(wav_grid, stokes_arrays, lam_settings_nm,
                                 fsr_nm=FSR_NM, finesse=FINESSE):
    """Sample Stokes profiles through the FP etalon at given settings.

    Args:
        wav_grid: Fine-grid wavelength array (nm).
        stokes_arrays: Iterable of (I, Q, U, V) on wav_grid.
        lam_settings_nm: Etalon centre wavelengths to sample.
        fsr_nm: Etalon FSR.
        finesse: Etalon finesse.

    Returns:
        ndarray of shape (4, len(lam_settings_nm)) with sampled
        I, Q, U, V values.
    """
    sampled = np.zeros((4, len(lam_settings_nm)))
    for i, lam in enumerate(lam_settings_nm):
        T = airy_transmission(wav_grid, lam, fsr_nm, finesse)
        norm = _trapz(T, wav_grid)
        for k, S in enumerate(stokes_arrays):
            sampled[k, i] = _trapz(T * S, wav_grid) / norm
    return sampled


# True-state ground truth.
B_true = 1500.0
gamma_true = np.deg2rad(50.0)
chi_true = np.deg2rad(35.0)
v_true_kms = 0.5

wav_fine = np.linspace(LAMBDA0_NM - 0.05, LAMBDA0_NM + 0.05, 2000)
I_t, Q_t, U_t, V_t = me_stokes_profiles(
    wav_fine, B_true, gamma_true, chi_true, v_los_kms=v_true_kms)

# Sample at the 6 SO/PHI wavelengths.
scan_lambdas = LAMBDA0_NM + np.array([-300.0, -140.0, -70.0, 0.0, 70.0, 140.0]) * 1e-4
sampled_clean = sample_stokes_through_etalon(
    wav_fine, (I_t, Q_t, U_t, V_t), scan_lambdas)

# Add 1e-3 I_c Gaussian noise.
noise_level = 1e-3
noise = np.random.normal(0.0, noise_level, sampled_clean.shape)
sampled_noisy = sampled_clean + noise

# Plot.
fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True)
stokes_names = ["I", "Q", "U", "V"]
stokes_continuous = [I_t, Q_t, U_t, V_t]
for k, (name, S_full) in enumerate(zip(stokes_names, stokes_continuous)):
    ax = axes[k]
    ax.plot(wav_fine, S_full, color="#212121", linewidth=1, alpha=0.5,
            label="continuous")
    ax.scatter(scan_lambdas, sampled_clean[k], color="#1976D2", s=60,
               zorder=5, label="clean sample")
    ax.errorbar(scan_lambdas, sampled_noisy[k], yerr=noise_level,
                fmt="o", color="#E91E63", markersize=5,
                label="noisy sample", zorder=4)
    ax.set_title("Stokes " + name)
    ax.set_xlabel("Wavelength [nm]")
    if k == 0:
        ax.set_ylabel("normalised intensity")
    ax.legend(fontsize=8)
fig.suptitle("6-position SO/PHI sampling: B={:.0f} G, gamma={:.0f} deg, chi={:.0f} deg".format(
    B_true, np.rad2deg(gamma_true), np.rad2deg(chi_true)), fontsize=12)
plt.tight_layout()
plt.show()

print("Maximum |V| signal: {:.3f}, noise floor: {:.3f}".format(
    np.max(np.abs(V_t)), noise_level))
print("=> SNR on V about {:.0f}".format(np.max(np.abs(V_t)) / noise_level))

## Part 4: 토이 Milne-Eddington 인버전 / Toy Milne-Eddington Inversion

SO/PHI는 **우주에서 RTE 인버전을 비행 중 수행한 사상 최초의 spectropolarimeter** 이다. 픽셀당 9 ME 파라미터 ($B, \gamma, \phi, v_{LOS}, \eta_0, \Delta\lambda_D, a, S_0, S_1$) 을 합성된 Stokes 프로파일에서 비선형 최소제곱으로 복원한다.

SO/PHI is the **first space spectropolarimeter to invert the RTE on-board**. Per pixel, 9 ME parameters ($B, \gamma, \phi, v_{LOS}, \eta_0, \Delta\lambda_D, a, S_0, S_1$) are recovered from the synthetic Stokes profiles via nonlinear least-squares.

이 셀은 Part 3에서 만든 잡음이 있는 6점 샘플로부터 4개의 핵심 자기적 파라미터 ($B, \gamma, \phi, v_{LOS}$) 를 복원하는 단순화된 인버전을 보여준다.

This cell performs a simplified inversion that recovers 4 key magnetic parameters ($B, \gamma, \phi, v_{LOS}$) from the noisy 6-point samples generated in Part 3.

In [ ]:
# =============================================================
# Toy ME Inversion via scipy.optimize.least_squares
# =============================================================

def forward_model_at_samples(params, lam_settings_nm, wav_fine_grid,
                             fsr_nm=FSR_NM, finesse=FINESSE):
    """Forward model: ME profiles -> etalon-weighted samples.

    Args:
        params: Tuple (B, gamma_deg, chi_deg, v_los_kms).
        lam_settings_nm: Etalon settings.
        wav_fine_grid: Fine integration grid.
        fsr_nm: FP free spectral range.
        finesse: FP finesse.

    Returns:
        Flat array of (I_samples, Q_samples, U_samples, V_samples).
    """
    B, gamma_deg, chi_deg, v_los = params
    I_m, Q_m, U_m, V_m = me_stokes_profiles(
        wav_fine_grid, B, np.deg2rad(gamma_deg), np.deg2rad(chi_deg),
        v_los_kms=v_los)
    sampled = sample_stokes_through_etalon(
        wav_fine_grid, (I_m, Q_m, U_m, V_m), lam_settings_nm,
        fsr_nm, finesse)
    return sampled.ravel()


def residuals(params, observed_flat, lam_settings_nm, wav_fine_grid):
    """Residual function for least_squares."""
    return forward_model_at_samples(params, lam_settings_nm, wav_fine_grid) - observed_flat


# Observed = noisy samples from Part 3.
observed_flat = sampled_noisy.ravel()

# Classical-estimation initial guess (paper Sect 7.4 default).
# B from V amplitude (weak-field), gamma from V/L ratio, chi from arctan(U/Q).
v_amplitude = np.max(np.abs(sampled_noisy[3]))
L_amplitude = np.sqrt(np.max(sampled_noisy[1] ** 2) + np.max(sampled_noisy[2] ** 2))
B_init = max(50.0, v_amplitude / 0.05 * 1000.0)  # Order-of-magnitude weak-field guess.
B_init = min(B_init, 3000.0)
gamma_init = np.rad2deg(np.arctan2(L_amplitude, max(v_amplitude, 1e-3)))
chi_init_rad = 0.5 * np.arctan2(
    np.sum(sampled_noisy[2]), np.sum(sampled_noisy[1]))
chi_init = np.rad2deg(chi_init_rad) % 180.0
v_init = 0.0

p0 = [B_init, gamma_init, chi_init, v_init]
bounds = ([10.0, 0.0, 0.0, -10.0], [4000.0, 180.0, 180.0, 10.0])

print("Initial guess (classical estimations):")
print("  B = {:.0f} G, gamma = {:.1f} deg, chi = {:.1f} deg, v = {:.2f} km/s".format(*p0))

result = least_squares(
    residuals, p0, bounds=bounds,
    args=(observed_flat, scan_lambdas, wav_fine),
    method="trf", max_nfev=200, ftol=1e-8)

B_fit, gamma_fit, chi_fit, v_fit = result.x

print("")
print("=" * 60)
print("ME inversion result:")
print("=" * 60)
header = "  {:<12s} {:>10s} {:>10s} {:>10s}".format("", "true", "recovered", "error")
print(header)
print("-" * 60)
row = "  {:<12s} {:>10.2f} {:>10.2f} {:>10.2f}"
print(row.format("B [G]", B_true, B_fit, B_fit - B_true))
print(row.format("gamma [deg]", np.rad2deg(gamma_true), gamma_fit,
                 gamma_fit - np.rad2deg(gamma_true)))
print(row.format("chi [deg]", np.rad2deg(chi_true), chi_fit,
                 chi_fit - np.rad2deg(chi_true)))
print(row.format("v_LOS [km/s]", v_true_kms, v_fit, v_fit - v_true_kms))
print("=" * 60)
print("  Iterations: {}, final cost: {:.2e}".format(result.nfev, result.cost))

# Plot fit overlaid on noisy data.
I_fit, Q_fit, U_fit, V_fit = me_stokes_profiles(
    wav_fine, B_fit, np.deg2rad(gamma_fit), np.deg2rad(chi_fit), v_los_kms=v_fit)
fit_continuous = [I_fit, Q_fit, U_fit, V_fit]

fig, axes = plt.subplots(1, 4, figsize=(16, 4), sharex=True)
for k, name in enumerate(stokes_names):
    ax = axes[k]
    ax.plot(wav_fine, stokes_continuous[k], color="#212121", linewidth=1.2,
            label="true", alpha=0.6)
    ax.plot(wav_fine, fit_continuous[k], color="#E91E63", linewidth=1.5,
            linestyle="--", label="recovered fit")
    ax.errorbar(scan_lambdas, sampled_noisy[k], yerr=noise_level,
                fmt="o", color="#1976D2", markersize=5, label="data", zorder=5)
    ax.set_title("Stokes " + name)
    ax.set_xlabel("Wavelength [nm]")
    if k == 0:
        ax.set_ylabel("normalised intensity")
    ax.legend(fontsize=8)
fig.suptitle("ME inversion fit vs ground truth", fontsize=12)
plt.tight_layout()
plt.show()

## Summary / 요약

이 노트북은 SO/PHI 4가지 핵심 측정 원리를 합성 데이터로 시연하였다 — 듀얼 망원경 광학, Fabry-Pérot 선 스캔, Zeeman 효과 하의 Stokes 프로파일 합성, 그리고 토이 Milne-Eddington 인버전. 실제 SO/PHI는 픽셀당 9 ME 파라미터를 비행 중 FPGA 에서 초당 3500 Stokes 프로파일 처리 속도로 복원하며, 이는 Solar Orbiter의 빠듯한 20 kbits/s 텔레메트리 한계를 극복한 핵심 설계 결정이다.

This notebook demonstrated SO/PHI's four core measurement principles via synthetic data — dual-telescope optics, Fabry-Pérot line scanning, Stokes profile synthesis under the Zeeman effect, and a toy Milne-Eddington inversion. The real SO/PHI recovers 9 ME parameters per pixel on-board at 3500 Stokes profiles/second on FPGAs — the key design choice that overcomes Solar Orbiter's tight 20 kbits/s telemetry budget.

| Concept / 개념 | This Paper / 이 논문 (SO/PHI 2020) | Modern / Future Equivalent |
|---|---|---|
| Dual telescope / 듀얼 망원경 | HRT (140 mm) + FDT (17.5 mm) on shared 2048² APS | DKIST has narrow-band Polarimeter; future Solar-C/EUVST inherits SO/PHI compact concept |
| Tunable narrow-band filter / 조정 가능 협대역 필터 | LiNbO3 FP etalon (FSR 0.301 nm, FWHM 106 mÅ) | DKIST VTF (PI-tuned air-spaced FP); ASO-S/FMG (ground-developed) |
| Polarisation modulator / 편광 변조기 | LCVR PMP (first space-qualified) | Hinode/SP rotating waveplate; SUNRISE/IMaX-balloon (LCVR heritage) |
| Inversion / 인버전 | On-board ME at 3500 profiles/s on FPGA | Ground-based VFISV (HMI), DeSIRe, neural-network inverters (Asensio Ramos & Díaz Baso 2019) |
| Spatial sampling at 0.28 AU / 0.28 AU 분해능 | HRT ~200 km/px | DKIST 20 km/px (ground 4 m); MUSE/SST ~50 km/px (1 m balloon) |

### 핵심 시사점 / Key Insights

- HRT는 0.28 AU 근일점에서 DKIST 4 m 지상 망원경에 견줄 수 있는 200 km 분해능을 14 cm 구경으로 달성한다 — 비결은 거리 자체이다 (1 AU 기준 14 cm 망원경의 회절 한계는 ~1 arcsec ≈ 720 km).
- HRT achieves 200 km resolution at 0.28 AU perihelion with only 14 cm aperture, comparable to a 4 m DKIST ground telescope — the trick is distance (a 14 cm telescope's diffraction limit at 1 AU is ~1 arcsec ≈ 720 km).
- LiNbO3 에탈론은 우주용 가변 협대역 필터로서 Michelson interferometer (MDI/HMI) 의 좁은 튜닝 한계를 극복하여 Solar Orbiter의 ±487 mÅ 도플러 스윙을 보상한다.
- The LiNbO3 etalon overcomes the narrow tuning of Michelson interferometers (MDI/HMI) and compensates Solar Orbiter's ±487 mÅ Doppler swing.
- 비행 중 ME 인버전은 raw Stokes 데이터를 ~5배 압축하며, 이는 텔레메트리 제한 우주 미션에서 분광편광계의 새 패러다임을 연다.
- On-board ME inversion compresses raw Stokes data by ~5×, opening a new paradigm for spectropolarimetry on telemetry-limited deep-space missions.

### References / 참고문헌

- Solanki, S. K., del Toro Iniesta, J. C., Woch, J., et al., "The Polarimetric and Helioseismic Imager on Solar Orbiter", A&A 642, A11 (2020). DOI: 10.1051/0004-6361/201935325.
- del Toro Iniesta, J. C., *Introduction to Spectropolarimetry*, Cambridge University Press (2003).
- Orozco Suárez, D. & Del Toro Iniesta, J. C., "The usefulness of analytic response functions", A&A 462, 1137 (2007).
- Martínez Pillet, V., del Toro Iniesta, J. C., Álvarez-Herrero, A., et al., "The Imaging Magnetograph eXperiment (IMaX) for the Sunrise balloon-borne solar observatory", Sol. Phys. 268, 57 (2011).